In [ ]:
from google.colab import drive
drive.mount('/content/drive')

! pip install corner
! pip install jaxopt
! pip install blackjax jax==0.7.2 jaxlib==0.7.2

import os
# os.environ["JAX_ENABLE_X64"] = "True"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
print(os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"])

path = '/content/drive/MyDrive/SchwarMAX-batch/'

import sys
sys.path.append(path)

from integrants_with_binning import assign_regular_grid
from model import *
from likelihoods import *
from utils import *
from sample_from_density import sample_from_density_grid
from CylindricalSpline import get_phi_m, evaluate_phi_axisymmetric

import os
# os.environ["JAX_ENABLE_X64"] = "True"

import jax
# jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import jax.numpy.linalg as jnn
import pandas as pd
import numpy as np
import scipy as sp
import pickle

import matplotlib.pyplot as plt

from constants import EPSILON

import blackjax
from blackjax.smc.resampling import systematic
import time

from tqdm import tqdm



def get_dict_data_bootstrap(path, filename, N_BOOTSTRAP = 100):

    with open(path + filename, 'rb') as f:
        bin_dict = pickle.load(f)

    X_minmax = jnp.array(bin_dict['X_minmax'])
    Y_minmax = jnp.array(bin_dict['Y_minmax'])
    nX_nY = jnp.array(bin_dict['nX_nY'])

    # voronoi binning mapping and data
    num_per_bin = jnp.array(bin_dict['num_per_bin'])
    total_bins = jnp.array(bin_dict['total_bins'])
    bin_mapping = jnp.array(bin_dict['bin_mapping'])
    surface_density = jnp.array(bin_dict['surface_density'])
    V_data = jnp.array(bin_dict['V_mean'])
    sigma_data = jnp.array(bin_dict['V_sigma'])
    h1_data = jnp.array(bin_dict['h1'])
    h2_data = jnp.array(bin_dict['h2'])
    h3_data = jnp.array(bin_dict['h3'])
    h4_data = jnp.array(bin_dict['h4'])
    v0 = jnp.array(bin_dict['v0'])
    s = jnp.array(bin_dict['s'])
    alpha, beta, gamma = bin_dict['orientation']


    XY_density_data_err = 0.01 * surface_density + EPSILON
    V_data_err = jnp.array(bin_dict['V_mean_err'])
    sigma_data_err = jnp.array(bin_dict['V_sigma_err'])
    h1_data_err = jnp.array(bin_dict['h1_err'])
    h2_data_err = jnp.array(bin_dict['h2_err'])
    h3_data_err = jnp.array(bin_dict['h3_err'])
    h4_data_err = jnp.array(bin_dict['h4_err'])

    '''
    Bootstrap the observation
    '''
    # rng = np.random.default_rng(42)
    # N_BOOTSTRAP = 100
    # y_xy_boot = np.array(surface_density[None, :] + rng.normal(size=(N_BOOTSTRAP, len(surface_density))) * XY_density_data_err[None, :])
    # y_h1_boot = np.array(h1_data[None, :] + rng.normal(size=(N_BOOTSTRAP, len(h1_data))) * h1_data_err[None, :])
    # y_h2_boot = np.array(h2_data[None, :] + rng.normal(size=(N_BOOTSTRAP, len(h2_data))) * h2_data_err[None, :])
    # y_h3_boot = np.array(h3_data[None, :] + rng.normal(size=(N_BOOTSTRAP, len(h3_data))) * h3_data_err[None, :])
    # y_h4_boot = np.array(h4_data[None, :] + rng.normal(size=(N_BOOTSTRAP, len(h4_data))) * h4_data_err[None, :])
    # V_boot = np.array(V_data[None, :] + rng.normal(size=(N_BOOTSTRAP, len(V_data))) * V_data_err[None, :])
    # sigma_boot = np.array(sigma_data[None, :] + rng.normal(size=(N_BOOTSTRAP, len(sigma_data))) * sigma_data_err[None, :])
    # # Fix the first one to always be the unperturbed system
    # y_xy_boot[0, :] = surface_density
    # y_h1_boot[0, :] = h1_data
    # y_h2_boot[0, :] = h2_data
    # y_h3_boot[0, :] = h3_data
    # y_h4_boot[0, :] = h4_data
    # V_boot[0, :] = V_data
    # sigma_boot[0, :] = sigma_data
    # y_xy_boot = jnp.array(y_xy_boot)
    # y_h1_boot = jnp.array(y_h1_boot)
    # y_h2_boot = jnp.array(y_h2_boot)
    # y_h3_boot = jnp.array(y_h3_boot)
    # y_h4_boot = jnp.array(y_h4_boot)
    # V_boot = jnp.array(V_boot)
    # sigma_boot = jnp.array(sigma_boot)

    rng = np.random.default_rng(42)
    XY_standard_normal = rng.normal(size=(N_BOOTSTRAP, len(surface_density)))
    h1_standard_normal = rng.normal(size=(N_BOOTSTRAP, len(h1_data)))
    h2_standard_normal = rng.normal(size=(N_BOOTSTRAP, len(h2_data)))
    h3_standard_normal = rng.normal(size=(N_BOOTSTRAP, len(h3_data)))
    h4_standard_normal = rng.normal(size=(N_BOOTSTRAP, len(h4_data)))
    V_standard_normal = rng.normal(size=(N_BOOTSTRAP, len(V_data)))
    sigma_standard_normal = rng.normal(size=(N_BOOTSTRAP, len(sigma_data)))
    XY_standard_normal[0, :] = 0.0
    h1_standard_normal[0, :] = 0.0
    h2_standard_normal[0, :] = 0.0
    h3_standard_normal[0, :] = 0.0
    h4_standard_normal[0, :] = 0.0
    V_standard_normal[0, :] = 0.0
    sigma_standard_normal[0, :] = 0.0
    XY_standard_normal = jnp.array(XY_standard_normal)
    h1_standard_normal = jnp.array(h1_standard_normal)
    h2_standard_normal = jnp.array(h2_standard_normal)
    h3_standard_normal = jnp.array(h3_standard_normal)
    h4_standard_normal = jnp.array(h4_standard_normal)
    V_standard_normal = jnp.array(V_standard_normal)
    sigma_standard_normal = jnp.array(sigma_standard_normal)

    from scipy.stats import qmc

    # with open(path + 'mock_axisymmetric_disc_Rzphi.pkl', 'rb') as f:
    #     Rzphi_density_data = pickle.load(f)
    # R_grid, z_grid, phi_grid = Rzphi_density_data['R_grid'], Rzphi_density_data['z_grid'], Rzphi_density_data['phi_grid']
    # dR = np.unique(R_grid)[1] - np.unique(R_grid)[0]
    # dz = np.unique(z_grid)[1] - np.unique(z_grid)[0]
    # dphi = np.unique(phi_grid)[1] - np.unique(phi_grid)[0]
    # sample_for_integration = Rzphi_density_data['sample_for_integration']

    R_min, R_max =0., 10.
    z_min, z_max = -3., 3.
    n_R, n_z, n_phi =10, 6, 10
    n_tot = int(n_R * n_z * n_phi)
    R_edge = jnp.linspace(R_min, R_max, n_R+1)
    z_edge = jnp.linspace(z_min, z_max, n_z+1)
    phi_edge = jnp.linspace(-jnp.pi, jnp.pi, n_phi+1)
    R_mids, z_mids, phi_mids = 0.5 * (R_edge[:-1] + R_edge[1:]), 0.5 * (z_edge[:-1] + z_edge[1:]), 0.5 * (phi_edge[:-1] + phi_edge[1:])
    dR, dz, dphi = R_edge[1]-R_edge[0], z_edge[1]-z_edge[0], phi_edge[1]-phi_edge[0]
    R_mids_mesh, z_mids_mesh, phi_mids_mesh = jnp.meshgrid(R_mids, z_mids, phi_mids, indexing='ij')
    Rzphi_mid_grid = jnp.stack([R_mids_mesh.ravel(), z_mids_mesh.ravel(), phi_mids_mesh.ravel()], axis=-1)  # (n_R*n_z*n_phi, 3)
    R_grid = Rzphi_mid_grid[:,0]
    z_grid = Rzphi_mid_grid[:,1]
    phi_grid = Rzphi_mid_grid[:,2]
    dR = np.unique(R_grid)[1] - np.unique(R_grid)[0]
    dz = np.unique(z_grid)[1] - np.unique(z_grid)[0]
    dphi = np.unique(phi_grid)[1] - np.unique(phi_grid)[0]
    Rzphi_minmax=jnp.array([[R_min, R_max],[z_min, z_max],[-jnp.pi, jnp.pi]])
    nRzphi=jnp.array([n_R,n_z,n_phi])
    num_segments_Rzphi=nRzphi.prod()
    Rzphi_strides = jnp.concatenate([jnp.array([1]), jnp.cumprod(nRzphi[:-1])])
    Rzphi_grid_indices = assign_regular_grid(Rzphi_mid_grid,
                                        grid_min=Rzphi_minmax[:,0],
                                        grid_max=Rzphi_minmax[:,1],
                                        n_bins=nRzphi,
                                        strides=Rzphi_strides)
    _, COUNTS = jnp.unique(Rzphi_grid_indices, return_counts=True)
    argsort = jnp.argsort(Rzphi_grid_indices)
    R_grid = R_grid[argsort]
    z_grid = z_grid[argsort]
    phi_grid = phi_grid[argsort]
    sampler = qmc.Sobol(d=3, scramble=False)
    sample_for_integration = sampler.random_base2(m=10)


    X_regular_grid, Y_regular_grid = bin_dict['X_regular_grid'], bin_dict['Y_regular_grid']
    dX = jnp.unique(X_regular_grid)[1] - jnp.unique(X_regular_grid)[0]
    dY = jnp.unique(Y_regular_grid)[1] - jnp.unique(Y_regular_grid)[0]
    sampler = qmc.Sobol(d=3, scramble=False)
    sample = sampler.random_base2(m=10)

    n_samples = 7500 #5_000  # Same number as original data
    x_grid = np.linspace(0., 12., 1000)
    logP_xexp = XexpX_pdf_log(x_grid, 4.0)
    key = jax.random.PRNGKey(10086)
    R_samples = sample_from_logP(x_grid, logP_xexp, n_samples, key)
    phi_samples = np.random.default_rng(42).uniform(0, 2*np.pi, size=n_samples)

    x_samples, y_samples = R_samples * np.cos(phi_samples), R_samples * np.sin(phi_samples)

    x_grid = np.linspace(0, 4, 1000)
    logP_exp = expX_pdf_log(x_grid, 1.5)
    key = jax.random.PRNGKey(10010)
    z_samples = sample_from_logP(x_grid, logP_exp, n_samples, key)
    w0 = np.array([
        x_samples,
        y_samples,
        z_samples,
    ]).T


    dict_data = {
        # 'w0': w0,
        'v0': v0,
        's': s,

        # 'Rzphi_density_data': Rzphi_density_data,
        'XY_density_data': surface_density,
        'XY_density_data_err': XY_density_data_err,
        'V_data': V_data,
        'V_data_err': V_data_err,
        'sigma_data': sigma_data,
        'sigma_data_err': sigma_data_err,
        'h1_data': h1_data,
        'h1_data_err': h1_data_err,
        'h2_data': h2_data,
        'h2_data_err': h2_data_err,
        'h3_data': h3_data,
        'h3_data_err': h3_data_err,
        'h4_data': h4_data,
        'h4_data_err': h4_data_err,
        'num_per_bin': num_per_bin,
        'bin_mapping': bin_mapping,
        'total_bins': total_bins.item(),

        # 'y_xy_boot': y_xy_boot,
        # 'y_h1_boot': y_h1_boot,
        # 'y_h2_boot': y_h2_boot,
        # 'y_h3_boot': y_h3_boot,
        # 'y_h4_boot': y_h4_boot,
        # 'V_boot': V_boot,
        # 'sigma_boot': sigma_boot,
        'XY_standard_normal': XY_standard_normal,
        'h1_standard_normal': h1_standard_normal,
        'h2_standard_normal': h2_standard_normal,
        'h3_standard_normal': h3_standard_normal,
        'h4_standard_normal': h4_standard_normal,
        'V_standard_normal': V_standard_normal,
        'sigma_standard_normal': sigma_standard_normal,

        'R_grid': R_grid,
        'z_grid': z_grid,
        'phi_grid': phi_grid,
        'R_minmax': [R_min, R_max],
        'z_minmax': [z_min, z_max],
        'phi_minmax': [-jnp.pi, jnp.pi],
        'Rzphi_n_tot': n_tot,
        'Rzphi_n_grid': jnp.array([n_R, n_z, n_phi]),
        'dR': dR,
        'dz': dz,
        'dphi': dphi,
        'sample_for_integration': sample_for_integration,

        'X_regular_grid': X_regular_grid,
        'Y_regular_grid': Y_regular_grid,
        'dX': dX,
        'dY': dY,
        'sample_for_integration_XY': sample,

        'X_minmax': X_minmax,
        'Y_minmax': Y_minmax,
        'nX_nY': nX_nY,


        'w0': w0,

        'alpha': alpha,
        'beta': beta,
        'gamma': gamma
    }

    return dict_data


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.4/172.4 kB 15.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of blackjax to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.3/172.3 kB 22.2 MB/s eta 0:00:00
  Attempting uninstall: jaxopt
    Found existing installation: jaxopt 0.8.5
    Uninstalling jaxopt-0.8.5:
      Successfully uninstalled jaxopt-0.8.5
0.95


In [ ]:
path = '/content/drive/MyDrive/SchwarMAX-batch/'
# filename = 'mock_Nbody_bar_XY_withRot_gal2_Nbins1000.pkl' #_Nbins1000
# beta_ls = [5,25,50,75]
# gamma_ls = [85,135,170]
beta_ls = [5,25,50,75]
gamma_ls = [60, 110, 140]
for beta_i in beta_ls:
  for gamma_i in gamma_ls:

    D = 50
    print(beta_i, gamma_i)

    filename = f'mock_data/mock_Nbody_bar_XY_withRot_Nbins600_beta{beta_i}_gamma{gamma_i}_D50_gal2.pkl' #_Nbins1000
    dict_data = get_dict_data_bootstrap(path, filename)

    with open(path + 'dict_phi_stellar_t_t0_7.pkl', 'rb') as f:
      d = pickle.load(f)
    dict_phi = {k: jnp.array(v) for k, v in d.items() if k != '_metadata'}

    dict_data['dict_phi'] = dict_phi
    # dict_data['alpha'] = 30.
    # dict_data['beta'] = 20.
    # dict_data['gamma'] = 140.


    ground_truth = [
        11.88,
        jnp.log10(19.2).item(),
        0.,
        1.5
    ]
    logL, _ = logl_fixed_potential_bootstrap(ground_truth, dict_data['dict_phi'], dict_data, dict_data['total_bins'], dict_data['Rzphi_n_tot'])
    print(logL)

    import time
    start = time.time()
    logL, _ = logl_fixed_potential_bootstrap(ground_truth, dict_data['dict_phi'], dict_data, dict_data['total_bins'], dict_data['Rzphi_n_tot'])
    logL.block_until_ready()  # Ensure computation finishes before timing
    end = time.time()
    print('time per logl evaluation', end - start, 's')
    print(logL)


    logl_fixed_potential_bootstrap_vmap = jax.vmap(logl_fixed_potential_bootstrap, in_axes=(0, None, None, None, None))

    # Add uncertainty to data
    XY_density_data, XY_density_data_err = dict_data['XY_density_data'], dict_data['XY_density_data_err']
    h1_data, h1_data_err = dict_data['h1_data'], dict_data['h1_data_err']
    h2_data, h2_data_err = dict_data['h2_data'], dict_data['h2_data_err']
    h3_data, h3_data_err = dict_data['h3_data'], dict_data['h3_data_err']
    h4_data, h4_data_err = dict_data['h4_data'], dict_data['h4_data_err']

    key = jax.random.PRNGKey(4008)
    key_chain = jax.random.split(key, num=5)
    noise_0 = jax.random.normal(key_chain[0], shape=XY_density_data.shape)
    noise_1 = jax.random.normal(key_chain[1], shape=h1_data.shape)
    noise_2 = jax.random.normal(key_chain[2], shape=h2_data.shape)
    noise_3 = jax.random.normal(key_chain[3], shape=h3_data.shape)
    noise_4 = jax.random.normal(key_chain[4], shape=h4_data.shape)
    XY_density_data_realisation = XY_density_data + noise_0 * XY_density_data_err
    h1_data_realisation = h1_data + noise_1 * h1_data_err
    h2_data_realisation = h2_data + noise_2 * h2_data_err
    h3_data_realisation = h3_data + noise_3 * h3_data_err
    h4_data_realisation = h4_data + noise_4 * h4_data_err
    dict_data['XY_density_data'] = XY_density_data_realisation
    dict_data['h1_data'] = h1_data_realisation
    dict_data['h2_data'] = h2_data_realisation
    dict_data['h3_data'] = h3_data_realisation
    dict_data['h4_data'] = h4_data_realisation


    # def log_prob(theta):
    #     params = [11.88, jnp.log10(19.2).item(), theta[0], theta[1]]
    #     ll, meff_data = logl_fixed_potential_bootstrap(params, dict_data['dict_phi'], dict_data, dict_data['total_bins'])  # convert from JAX array
    #     if not np.isfinite(ll):
    #         return -np.inf
    #     return ll, meff_data#, meff_model
    def log_prob_batch(theta):
        params = jnp.array(theta)
        ll, meff_data = logl_fixed_potential_bootstrap_vmap(params, dict_data['dict_phi'], dict_data, dict_data['total_bins'], dict_data['Rzphi_n_tot'])  # convert from JAX array
        ll = np.where(np.isfinite(ll), ll, -np.inf)
        return ll, meff_data#, meff_model

    ground_truth = [
        0.,
        1.5
    ]

    grid_logM2L = np.linspace(-0.6,0.6,30)
    grid_Omega = np.linspace(10,50,30)

    param_grid1 = -grid_logM2L # log10(L/M)
    param_grid2 = np.log10(grid_Omega) # log10(Omega)
    Grid1, Grid2 = np.meshgrid(param_grid1, param_grid2)

    N_grid = len(Grid1.flatten())
    logM_halo_grid = np.ones(N_grid) * 11.88
    logRs_halo_grid = np.ones(N_grid) * np.log10(19.2).item()
    param_grid = np.array([logM_halo_grid, logRs_halo_grid, Grid1.flatten(), Grid2.flatten()]).T
    # param_grid[:, 0] = (param_grid[:, 0] - 0.5) * 1 + ground_truth[0]
    # param_grid[:, 1] = (param_grid[:, 1] - 0.5) * 0.75 + ground_truth[1]

    Chunk_size = 12
    Chunk_total = (len(param_grid) // Chunk_size) + 1

    from tqdm import tqdm
    log_prob_grid = jnp.array([])
    m_eff_data_grid = jnp.array([])
    m_eff_model_grid = jnp.array([])
    log_L2M_grid = jnp.array([])
    log_Omega_grid = jnp.array([])
    for i in tqdm(range(Chunk_total)):
      params_i = param_grid[i*Chunk_size:(i+1)*Chunk_size]

      # logl, meff_data = log_prob(param_grid[i])
      logl, meff_data = log_prob_batch(params_i)

      log_prob_grid = jnp.append(log_prob_grid, logl)
      m_eff_data_grid = jnp.append(m_eff_data_grid, meff_data)
      log_L2M_grid = jnp.append(log_L2M_grid, params_i[:, 2])
      log_Omega_grid = jnp.append(log_Omega_grid, params_i[:, 3])
      # log_L2M_grid.append(param_grid[i, 0])
      # log_Omega_grid.append(param_grid[i, 1])
      # log_prob_grid.append(logl)
      # m_eff_data_grid.append(meff_data)
      # # m_eff_model_grid.append(meff_model)
      if i % 1 == 0:
        pd.DataFrame({
          'log_light_to_mass_ratio': log_L2M_grid,
          'log_Omega': log_Omega_grid,
          'log_prob': log_prob_grid,
          'meff_data': m_eff_data_grid,
          # 'meff_model': m_eff_model_grid
        }).to_csv(path + f'/grid_search_result_0422_beta{beta_i}_gamma{gamma_i}_D50_gal2.csv', index=False)

    log_L2M_grid = np.array(log_L2M_grid)
    log_Omega_grid = np.array(log_Omega_grid)
    log_prob_grid = np.array(log_prob_grid)
    m_eff_data_grid = np.array(m_eff_data_grid)
    # m_eff_model_grid = np.array(m_eff_model_grid)

    pd.DataFrame({
        'log_light_to_mass_ratio': log_L2M_grid,
        'log_Omega': log_Omega_grid,
        'log_prob': log_prob_grid,
        'meff_data': m_eff_data_grid,
        # 'meff_model': m_eff_model_grid
    }).to_csv(path + f'/grid_search_result_0422_beta{beta_i}_gamma{gamma_i}_D50_gal2.csv', index=False)

5 60
-8796.368
time per logl evaluation 5.7358880043029785 s
-8797.006


100%|██████████| 76/76 [1:03:29<00:00, 50.12s/it]


5 110
-9403.333
time per logl evaluation 5.856734991073608 s
-9402.842


100%|██████████| 76/76 [1:03:58<00:00, 50.50s/it]


5 140
-7921.4653
time per logl evaluation 5.818951368331909 s
-7921.524


...

100%|██████████| 76/76 [1:02:23<00:00, 49.25s/it]


75 110
-9516.027
time per logl evaluation 5.851341485977173 s
-9518.312


100%|██████████| 76/76 [1:04:30<00:00, 50.93s/it]


75 140
-10194.984
time per logl evaluation 5.853703498840332 s
-10194.671


100%|██████████| 76/76 [1:05:06<00:00, 51.40s/it]


In [ ]:
import pickle

path = '/content/drive/MyDrive/SchwarMAX-batch/'
# filename = 'mock_Nbody_bar_XY_withRot_gal2_Nbins1000.pkl' #_Nbins1000
filename = 'mock_Nbody_bar_XY_withRot_Nbins600_beta25_gamma170.pkl' #_Nbins1000
dict_data = get_dict_data_bootstrap(path, filename)

with open(path + 'dict_phi_stellar_t_t0_4.pkl', 'rb') as f:
  d = pickle.load(f)
dict_phi = {k: jnp.array(v) for k, v in d.items() if k != '_metadata'}

dict_data['dict_phi'] = dict_phi
# dict_data['alpha'] = 30.
# dict_data['beta'] = 20.
# dict_data['gamma'] = 140.

params_halo = [
    11.88,
    jnp.log10(19.2).item()
    ]

ground_truth = [
    0.,
    1.5,
    0.5
]
logL, _ = logl_fixed_potential_bootstrap_amp(ground_truth, dict_data['dict_phi'], params_halo, dict_data, dict_data['total_bins'], dict_data['Rzphi_n_tot'])
print(logL)

import time
start = time.time()
logL, _ = logl_fixed_potential_bootstrap_amp(ground_truth, dict_data['dict_phi'], params_halo, dict_data, dict_data['total_bins'], dict_data['Rzphi_n_tot'])
logL.block_until_ready()  # Ensure computation finishes before timing
end = time.time()
print('time per logl evaluation', end - start, 's')
print(logL)

-264.42374
time per logl evaluation 3.550402879714966 s
-264.45392


In [ ]:
path = '/content/drive/MyDrive/SchwarMAX-batch/'

N_CHAINS = 16              # number of parallel chains (must fit in GPU vmap)
N_STEPS = 1000             # total MCMC steps per chain
CHECKPOINT_EVERY = 10
BURNIN = 200
ADAPT_EVERY = 100          # re-estimate covariance every N steps
ADAPT_AFTER = 150          # delay adaptation until chains have explored


CHECKPOINT_FILE = os.path.join(path, 'mcmc_checkpoint_0413_beta25_gamma170_fixedpot.pkl')
OUTPUT_FILE = os.path.join(path, 'mcmc_results_0413_beta25_gamma170_fixedpot.pkl')
OUTPUT_CSV = os.path.join(path, 'mcmc_posterior_0413_beta25_gamma170_fixedpot.csv')


# ── Load data ────────────────────────────────────────────────────────
num_Vbin = int(dict_data['total_bins'])
params_halo = [
    11.88,
    jnp.log10(19.2).item()
    ]

# -- Initial Guess ----------
res = jnp.array([
    -0.2, 1.5, 0.5
])

# ── Prior bounds (13D) ───────────────────────────────────────────────
NDIM = 3
param_names = [
    'log_light_to_mass_ratio', 'log_Omega', 'log_sigma',
]

BOUNDS_LO = jnp.array([
  -2., 0., -1.,
])
BOUNDS_HI = jnp.array([
  2., 2., 2.,
])

# ── Log-posterior ────────────────────────────────────────────────────
def logdensity_fn(theta):
    in_bounds = jnp.all((theta >= BOUNDS_LO) & (theta <= BOUNDS_HI))
    log_vol = jnp.sum(jnp.log(BOUNDS_HI - BOUNDS_LO))
    logprior = jnp.where(in_bounds, -log_vol, -jnp.inf)

    ll, _ = logl_fixed_potential_bootstrap_amp(theta, dict_data['dict_phi'], params_halo, dict_data, dict_data['total_bins'])
    ll = jnp.where(jnp.isfinite(ll), ll, -1e30)

    return logprior + ll

# ── Initial proposal (small for ~30-40% acceptance) ──────────────────
OPTIMAL_SCALE = 2.38 / np.sqrt(NDIM)  # Roberts & Rosenthal 2001

rmh_sigma_init = jnp.array([
  0.3, 0.3, 0.5
])

_proposal_init = jnp.array([
  0.05, 0.05, 0.05
])

# ── Helper: build sampler from proposal ──────────────────────────────
def build_sampler(proposal):
    rw = blackjax.additive_step_random_walk(logdensity_fn, proposal)
    return jax.vmap(rw.step), jax.vmap(rw.init)

N_HALF = N_CHAINS // 2  # each vmap batch size

def _split_pytree(tree):
    """Split a pytree of (N_CHAINS, ...) arrays into two halves."""
    return (jax.tree.map(lambda x: x[:N_HALF], tree),
            jax.tree.map(lambda x: x[N_HALF:], tree))

def _merge_pytree(tree0, tree1):
    """Merge two pytree halves back into (N_CHAINS, ...)."""
    return jax.tree.map(lambda a, b: jnp.concatenate([a, b], axis=0), tree0, tree1)

def batched_init(vmap_init, positions):
    """Initialise states in two batches to avoid OOM."""
    states_0 = vmap_init(positions[:N_HALF])
    states_1 = vmap_init(positions[N_HALF:])
    return _merge_pytree(states_0, states_1)

def batched_step(vmap_step, keys, states):
    """Run one MCMC step in two batches to avoid OOM."""
    states_0, states_1 = _split_pytree(states)
    keys_0, keys_1 = keys[:N_HALF], keys[N_HALF:]
    new_states_0, infos_0 = vmap_step(keys_0, states_0)
    new_states_1, infos_1 = vmap_step(keys_1, states_1)
    return _merge_pytree(new_states_0, new_states_1), _merge_pytree(infos_0, infos_1)

# ── Initial positions ────────────────────────────────────────────────
def make_init_positions(rng_key):
    """Start chains near best-fit with small perturbation."""
    p0 = jnp.array(res)
    noise = jax.random.normal(rng_key, shape=(N_CHAINS, NDIM)) * rmh_sigma_init[None, :]
    positions = p0[None, :] + noise
    positions = jnp.clip(positions, BOUNDS_LO[None, :], BOUNDS_HI[None, :])
    return positions


# ── Checkpointing ───────────────────────────────────────────────────
def save_checkpoint(all_samples, all_logprob, step, rng_key, adapt_count=0,
                    proposal_L=None, scale_factor=1.0):
    ckpt = {
        'all_samples': [np.array(s) for s in all_samples],
        'all_logprob': [np.array(lp) for lp in all_logprob],
        'step': step,
        'rng_key': np.array(rng_key),
        'adapt_count': adapt_count,
        'proposal_L': np.array(proposal_L) if proposal_L is not None else None,
        'scale_factor': scale_factor,
    }
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(ckpt, f)


def load_checkpoint():
    if not os.path.exists(CHECKPOINT_FILE):
        return None
    with open(CHECKPOINT_FILE, 'rb') as f:
        ckpt = pickle.load(f)
    n_steps = len(ckpt['all_samples'])
    n_chains = ckpt['all_samples'][0].shape[0]
    print(f"Found checkpoint: {n_steps} steps, {n_chains} chains, "
          f"adapt_count={ckpt.get('adapt_count', 0)}")
    return ckpt


# ── Main loop ────────────────────────────────────────────────────────
def run_mcmc(resume=True):
    rng_key = jax.random.PRNGKey(42)

    ckpt = load_checkpoint() if resume else None

    if ckpt is not None:
        # ── Resume from checkpoint ────────────────────────────────
        all_positions = ckpt['all_samples']
        all_logprob = ckpt['all_logprob']
        start_step = ckpt['step']
        rng_key = jnp.array(ckpt['rng_key'])
        adapt_count = ckpt.get('adapt_count', 0)
        proposal_L = ckpt.get('proposal_L', None)
        scale_factor = ckpt.get('scale_factor', 1.0)

        # Reconstruct states from last saved positions/logprob
        last_positions = jnp.array(all_positions[-1])
        if proposal_L is not None:
            proposal = blackjax.mcmc.random_walk.normal(
                jnp.array(proposal_L * scale_factor))
        else:
            proposal = blackjax.mcmc.random_walk.normal(_proposal_init)
        vmap_step, vmap_init = build_sampler(proposal)
        states = batched_init(vmap_init, last_positions)

        print(f"Resumed from step {start_step}, {len(all_positions)} samples, "
              f"adapt_count={adapt_count}, scale_factor={scale_factor:.3f}")
    else:
        # ── Fresh start ───────────────────────────────────────────
        rng_key, init_key = jax.random.split(rng_key)
        positions = make_init_positions(init_key)

        print(f"Initialising {N_CHAINS} chains in 2 batches of {N_HALF}...")
        proposal_init = blackjax.mcmc.random_walk.normal(_proposal_init)
        vmap_step, vmap_init = build_sampler(proposal_init)
        states = batched_init(vmap_init, positions)
        print(f"  Init done.")
        all_positions = []
        all_logprob = []
        start_step = 0
        adapt_count = 0
        proposal_L = None
        scale_factor = 1.0

    # Track recent acceptance (last ADAPT_EVERY steps)
    recent_accepts = []

    print(f"\nAdaptive RMH: {N_CHAINS} chains, {N_STEPS} steps, {NDIM}D")
    print(f"  Adapt covariance every {ADAPT_EVERY} steps after step {ADAPT_AFTER}")
    print(f"  Using all samples from step {ADAPT_AFTER} onward for covariance")
    print(f"  Optimal scale: {OPTIMAL_SCALE:.3f}")
    print(f"  Starting from step {start_step + 1}")

    pbar = tqdm(range(start_step + 1, N_STEPS + 1), desc="RMH", unit="step")
    for step in pbar:
        rng_key, step_key = jax.random.split(rng_key)
        keys = jax.random.split(step_key, N_CHAINS)

        states, infos = batched_step(vmap_step, keys, states)

        all_positions.append(np.array(states.position))
        all_logprob.append(np.array(states.logdensity))

        step_accept = float(jnp.mean(infos.acceptance_rate))
        recent_accepts.append(step_accept)
        if len(recent_accepts) > ADAPT_EVERY:
            recent_accepts.pop(0)

        # ── Adapt proposal covariance ────────────────────────────
        if step >= ADAPT_AFTER and step % ADAPT_EVERY == 0:
            # Use samples from ADAPT_AFTER onward (skip burn-in)
            samples_post_burnin = np.stack(all_positions[int(ADAPT_AFTER-50):], axis=0)
            flat = samples_post_burnin.reshape(-1, NDIM)
            n_used = flat.shape[0]

            if n_used > 2 * NDIM:
                cov = np.cov(flat.T)
                diag = np.diag(np.diag(cov))
                cov_reg = 0.8 * cov + 0.2 * diag
                proposal_cov = OPTIMAL_SCALE**2 * cov_reg
                try:
                    L = np.linalg.cholesky(proposal_cov)
                    proposal_L = L

                    recent_acc = np.mean(recent_accepts)

                    proposal = blackjax.mcmc.random_walk.normal(
                        jnp.array(L * scale_factor))
                    vmap_step, _ = build_sampler(proposal)
                    adapt_count += 1

                    tqdm.write(
                        f"  [Adapt #{adapt_count} at step {step}] "
                        f"recent_acc={recent_acc:.3f}, scale={scale_factor:.3f}, "
                        f"n={n_used}, "
                        f"cov diag={np.sqrt(np.diag(proposal_cov))[:3].round(4)}")
                except np.linalg.LinAlgError:
                    tqdm.write(f"  [Adapt #{adapt_count+1} at step {step}] "
                               f"Cholesky failed, keeping current proposal")
            else:
                tqdm.write(f"  [Adapt at step {step}] "
                           f"Too few samples ({n_used}), skipping")

        if step % 10 == 0:
            recent_acc = np.mean(recent_accepts) if recent_accepts else 0.0
            pbar.set_postfix(
                logP=f"{float(jnp.mean(states.logdensity)):.1f}",
                acc=f"{recent_acc:.3f}",
                adapt=adapt_count,
            )

        if step % CHECKPOINT_EVERY == 0:
            save_checkpoint(all_positions, all_logprob, step, rng_key,
                            adapt_count, proposal_L, scale_factor)

    # ── Results ──────────────────────────────────────────────────────
    chain = np.stack(all_positions, axis=0)  # (N_STEPS, N_CHAINS, NDIM)
    logprob = np.stack(all_logprob, axis=0)
    print(f"\nChain shape: {chain.shape}")

    # Flat samples after burn-in
    if chain.shape[0] > BURNIN:
        flat_samples = chain[BURNIN:].reshape(-1, NDIM)
    else:
        flat_samples = chain.reshape(-1, NDIM)

    print(f"\n── Posterior summary ({flat_samples.shape[0]} samples) ──")
    print(f"{'Parameter':>30s} {'mean':>10s} {'std':>10s} "
          f"{'2.5%':>10s} {'97.5%':>10s} {'truth':>10s}")
    print("-" * 82)
    for i, name in enumerate(param_names):
        s = flat_samples[:, i]
        truth = res[i]
        print(f"{name:>30s} {s.mean():10.4f} {s.std():10.4f} "
              f"{np.percentile(s, 2.5):10.4f} {np.percentile(s, 97.5):10.4f} "
              f"{truth:10.4f}")

    # Save
    results = {
        'chain': chain,
        'logprob': logprob,
        'flat_samples': flat_samples,
        'param_names': param_names,
    }
    with open(OUTPUT_FILE, 'wb') as f:
        pickle.dump(results, f)
    print(f"\nResults saved to {OUTPUT_FILE}")

    pd.DataFrame(flat_samples, columns=param_names).to_csv(OUTPUT_CSV, index=False)
    print(f"CSV saved to {OUTPUT_CSV}")

    return chain, logprob


if __name__ == '__main__':
    run_mcmc(resume = True)


Initialising 16 chains in 2 batches of 8...
  Init done.

Adaptive RMH: 16 chains, 1000 steps, 3D
  Adapt covariance every 100 steps after step 150
  Using all samples from step 150 onward for covariance
  Optimal scale: 1.374
  Starting from step 1


RMH:  20%|██        | 200/1000 [2:04:55<8:19:46, 37.48s/step, acc=0.021, adapt=1, logP=502.0]

  [Adapt #1 at step 200] recent_acc=0.021, scale=1.000, n=1600, cov diag=[0.0676 0.1789 0.047 ]


RMH:  30%|███       | 300/1000 [3:07:23<7:17:25, 37.49s/step, acc=0.008, adapt=2, logP=562.1]

  [Adapt #2 at step 300] recent_acc=0.008, scale=1.000, n=3200, cov diag=[0.0602 0.1356 0.0379]


...

  [Adapt #5 at step 600] recent_acc=0.005, scale=1.000, n=8000, cov diag=[0.0446 0.0864 0.0293]


RMH:  70%|███████   | 700/1000 [7:17:14<3:07:21, 37.47s/step, acc=0.002, adapt=6, logP=572.9]

  [Adapt #6 at step 700] recent_acc=0.002, scale=1.000, n=9600, cov diag=[0.0419 0.0789 0.0277]


RMH:  78%|███████▊  | 785/1000 [8:10:38<2:14:22, 37.50s/step, acc=0.001, adapt=6, logP=573.5]


KeyboardInterrupt: 

In [ ]:

# ── Configuration ────────────────────────────────────────────────────
path = '/content/drive/MyDrive/SchwarMAX-batch/'

N_WALKERS = 16             # must be even and >= 2*NDIM
N_STEPS = 1000
CHECKPOINT_EVERY = 10
BURNIN = 300
STRETCH_A = 2.0            # stretch move scale parameter

CHECKPOINT_FILE = os.path.join(path, 'ensemble_checkpoint_0413_beta25_gamma170_fixedpot.pkl')
OUTPUT_FILE = os.path.join(path, 'ensemble_results_0413_beta25_gamma170_fixedpot.pkl')
OUTPUT_CSV = os.path.join(path, 'ensemble_posterior_0413_beta25_gamma170_fixedpot.csv')

# ── Load data ────────────────────────────────────────────────────────
num_Vbin = int(dict_data['total_bins'])
params_halo = [
    11.88,
    jnp.log10(19.2).item()
    ]

# -- Initial Guess ----------
res = jnp.array([
    -0.2, 1.5, 0.5
])

# ── Prior bounds (13D) ───────────────────────────────────────────────
NDIM = 3
param_names = [
    'log_light_to_mass_ratio', 'log_Omega', 'log_sigma',
]

BOUNDS_LO = jnp.array([
  -2., 0., -1.,
])
BOUNDS_HI = jnp.array([
  2., 2., 2.,
])

# ── Log-posterior ────────────────────────────────────────────────────
def logdensity_fn(theta):
    in_bounds = jnp.all((theta >= BOUNDS_LO) & (theta <= BOUNDS_HI))
    log_vol = jnp.sum(jnp.log(BOUNDS_HI - BOUNDS_LO))
    logprior = jnp.where(in_bounds, -log_vol, -jnp.inf)

    ll, _ = logl_fixed_potential_bootstrap_amp(theta, dict_data['dict_phi'], params_halo, dict_data, dict_data['total_bins'])
    ll = jnp.where(jnp.isfinite(ll), ll, -1e30)

    return logprior + ll

_vmap_logdensity = jax.vmap(logdensity_fn)

print('begining logL', logl_fixed_potential_bootstrap_amp(res, dict_data['dict_phi'], params_halo, dict_data, dict_data['total_bins']))
# ── Move weights (emcee-style mixture) ──────────────────────────────
# Same mix as the user's emcee config:
#   70% DEMove, 20% DESnookerMove, 10% StretchMove
MOVE_WEIGHTS = jnp.array([0.7, 0.9, 1.0])  # cumulative thresholds

DE_GAMMA = 2.38 / jnp.sqrt(2.0 * NDIM)    # ter Braak (2006) optimal
DE_NOISE = 1e-5                              # small jitter for ergodicity
SNOOKER_GAMMA = 1.7                          # ter Braak & Vrugt (2008)


# ── Stretch move ────────────────────────────────────────────────────
def _sample_z(rng_key, n, a=STRETCH_A):
    """Sample Z from g(z) ∝ 1/sqrt(z) on [1/a, a]."""
    u = jax.random.uniform(rng_key, shape=(n,))
    return ((a - 1.0) * u + 1.0) ** 2 / a


def _stretch_propose(rng_key, active, complement, ndim):
    """Stretch move (Goodman & Weare 2010).
    Returns (proposals, log_factors)."""
    n_half = active.shape[0]
    k1, k2 = jax.random.split(rng_key)

    idx = jax.random.randint(k1, (n_half,), 0, complement.shape[0])
    companions = complement[idx]
    z = _sample_z(k2, n_half)

    proposals = companions + z[:, None] * (active - companions)
    log_factors = (ndim - 1) * jnp.log(z)
    return proposals, log_factors


# ── DE move (ter Braak 2006) ────────────────────────────────────────
def _de_propose(rng_key, active, complement, ndim,
                gamma=None, sigma=DE_NOISE):
    """Differential Evolution move.
    Y = X + gamma * (c[r1] - c[r2]) + noise
    Symmetric proposal → log_factor = 0."""
    n_half = active.shape[0]
    n_comp = complement.shape[0]
    k1, k2, k3 = jax.random.split(rng_key, 3)

    if gamma is None:
        gamma = 2.38 / jnp.sqrt(2.0 * ndim)

    # Pick two distinct companions per walker
    r1 = jax.random.randint(k1, (n_half,), 0, n_comp)
    r2 = jax.random.randint(k2, (n_half,), 0, n_comp - 1)
    # Avoid r1 == r2: shift r2 up by 1 where r2 >= r1
    r2 = jnp.where(r2 >= r1, r2 + 1, r2)

    diff = complement[r1] - complement[r2]
    noise = sigma * jax.random.normal(k3, active.shape)

    proposals = active + gamma * diff + noise
    log_factors = jnp.zeros(n_half)  # symmetric proposal
    return proposals, log_factors


# ── DE-Snooker move (ter Braak & Vrugt 2008) ───────────────────────
def _snooker_propose(rng_key, active, complement, ndim,
                     gamma=SNOOKER_GAMMA):
    """DE-Snooker move: project differential onto line through walker and pivot.
    Has Jacobian factor (||Y-z0|| / ||X-z0||)^(D-1)."""
    n_half = active.shape[0]
    n_comp = complement.shape[0]
    k1, k2, k3 = jax.random.split(rng_key, 3)

    # Pick 3 companions: z0 (pivot), z1, z2 (for differential)
    r0 = jax.random.randint(k1, (n_half,), 0, n_comp)
    r12 = jax.random.randint(k2, (n_half, 2), 0, n_comp)
    r1, r2 = r12[:, 0], r12[:, 1]

    z0 = complement[r0]   # pivot
    z1 = complement[r1]
    z2 = complement[r2]

    # Direction: active → pivot
    direction = active - z0                           # (n_half, ndim)
    dist = jnp.linalg.norm(direction, axis=1, keepdims=True)
    dist_safe = jnp.maximum(dist, 1e-30)
    d_hat = direction / dist_safe                     # unit vector

    # Differential projected onto direction
    diff = z1 - z2                                     # (n_half, ndim)
    proj = jnp.sum(diff * d_hat, axis=1, keepdims=True)  # scalar projection

    # Propose along the direction
    proposals = active + gamma * proj * d_hat

    # Jacobian: (||Y - z0|| / ||X - z0||)^(D-1)
    new_dist = jnp.linalg.norm(proposals - z0, axis=1)
    log_factors = (ndim - 1) * jnp.log(new_dist / dist_safe.squeeze())

    return proposals, log_factors


# ── Mixed move: per-walker random selection ─────────────────────────
def _mixed_move_half(rng_key, active, active_logp, complement):
    """Update active walkers using a mixture of DE, Snooker, and Stretch moves.

    Each walker independently draws which move to use, then all proposals
    are evaluated in a single vmapped logL batch.
    """
    n_half = active.shape[0]
    k_sel, k_de, k_snk, k_str, k_acc = jax.random.split(rng_key, 5)

    # Generate proposals from ALL three moves (compute all, select per-walker)
    prop_de, lf_de = _de_propose(k_de, active, complement, NDIM)
    prop_snk, lf_snk = _snooker_propose(k_snk, active, complement, NDIM)
    prop_str, lf_str = _stretch_propose(k_str, active, complement, NDIM)

    # Per-walker move selection
    u_sel = jax.random.uniform(k_sel, (n_half,))
    use_de = u_sel < MOVE_WEIGHTS[0]                                    # 0.0–0.7
    use_snk = (u_sel >= MOVE_WEIGHTS[0]) & (u_sel < MOVE_WEIGHTS[1])   # 0.7–0.9
    # else: stretch                                                      # 0.9–1.0

    # Select proposal and log_factor per walker
    proposals = jnp.where(use_de[:, None], prop_de,
                    jnp.where(use_snk[:, None], prop_snk, prop_str))
    log_factors = jnp.where(use_de, lf_de,
                    jnp.where(use_snk, lf_snk, lf_str))

    # Evaluate all proposals in one vmapped batch
    prop_logp = _vmap_logdensity(proposals)

    # Metropolis acceptance
    log_accept = log_factors + prop_logp - active_logp
    log_u = jnp.log(jax.random.uniform(k_acc, (n_half,)))
    accept = log_u < log_accept

    new_active = jnp.where(accept[:, None], proposals, active)
    new_logp = jnp.where(accept, prop_logp, active_logp)
    n_accepted = jnp.sum(accept)

    return new_active, new_logp, n_accepted


def ensemble_step(rng_key, positions, logp):
    """One full ensemble step: update both halves with mixed moves.

    Args:
        rng_key: JAX PRNG key
        positions: (N_WALKERS, NDIM)
        logp: (N_WALKERS,)

    Returns:
        new_positions, new_logp, n_accepted
    """
    n_half = N_WALKERS // 2
    k1, k2 = jax.random.split(rng_key)

    s0, s1 = positions[:n_half], positions[n_half:]
    lp0, lp1 = logp[:n_half], logp[n_half:]

    # Phase 1: update S0 using S1
    s0, lp0, acc0 = _mixed_move_half(k1, s0, lp0, s1)

    # Phase 2: update S1 using updated S0
    s1, lp1, acc1 = _mixed_move_half(k2, s1, lp1, s0)

    new_positions = jnp.concatenate([s0, s1], axis=0)
    new_logp = jnp.concatenate([lp0, lp1], axis=0)
    n_accepted = acc0 + acc1

    return new_positions, new_logp, n_accepted


# ── Initial positions ────────────────────────────────────────────────
def make_init_positions(rng_key):
    """Start walkers spread across a broad region around the best-fit.

    Uses ~30% of the prior width per parameter so walkers explore widely
    from the start — the starting point may not be the true mode.
    """
    p0 = jnp.array(res)
    # Spread = 10% of prior half-width per parameter
    init_spread = 0.1 * jnp.ones(len(BOUNDS_HI))# * (BOUNDS_HI - BOUNDS_LO) / 2.0
    noise = jax.random.normal(rng_key, shape=(N_WALKERS, NDIM)) * init_spread[None, :]
    positions = p0[None, :] + noise
    positions = jnp.clip(positions, BOUNDS_LO[None, :], BOUNDS_HI[None, :])
    return positions


# ── Checkpointing ───────────────────────────────────────────────────
def save_checkpoint(all_positions, all_logprob, step, rng_key):
    ckpt = {
        'all_samples': [np.array(s) for s in all_positions],
        'all_logprob': [np.array(lp) for lp in all_logprob],
        'step': step,
        'rng_key': np.array(rng_key),
    }
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(ckpt, f)


def load_checkpoint():
    if not os.path.exists(CHECKPOINT_FILE):
        return None
    with open(CHECKPOINT_FILE, 'rb') as f:
        ckpt = pickle.load(f)
    n_steps = len(ckpt['all_samples'])
    n_walkers = ckpt['all_samples'][0].shape[0]
    print(f"Found checkpoint: {n_steps} steps, {n_walkers} walkers")
    return ckpt


# ── Main loop ────────────────────────────────────────────────────────
def run_ensemble(resume=True):
    rng_key = jax.random.PRNGKey(42)

    ckpt = load_checkpoint() if resume else None

    if ckpt is not None:
        all_positions = ckpt['all_samples']
        all_logprob = ckpt['all_logprob']
        start_step = ckpt['step']
        rng_key = jnp.array(ckpt['rng_key'])

        positions = jnp.array(all_positions[-1])
        logp = jnp.array(all_logprob[-1])

        print(f"Resumed from step {start_step}, {len(all_positions)} samples")
    else:
        rng_key, init_key = jax.random.split(rng_key)
        positions = make_init_positions(init_key)

        print(f"Initialising {N_WALKERS} walkers...")
        n_half = N_WALKERS // 2
        logp = jnp.concatenate([
            _vmap_logdensity(positions[:n_half]),
            _vmap_logdensity(positions[n_half:]),
        ])
        print(f"  Init done. logP: mean={float(jnp.mean(logp)):.1f}, "
              f"max={float(jnp.max(logp)):.1f}")

        all_positions = [np.array(positions)]
        all_logprob = [np.array(logp)]
        start_step = 0

    print(f"\nEnsemble MCMC: {N_WALKERS} walkers, {N_STEPS} steps, {NDIM}D")
    print(f"  Stretch parameter a={STRETCH_A}")
    print(f"  Starting from step {start_step + 1}")

    pbar = tqdm(range(start_step + 1, N_STEPS + 1), desc="Ensemble", unit="step")
    for step in pbar:
        rng_key, step_key = jax.random.split(rng_key)

        positions, logp, n_accepted = ensemble_step(step_key, positions, logp)

        all_positions.append(np.array(positions))
        all_logprob.append(np.array(logp))

        if step % 5 == 0:
            acc = float(n_accepted) / N_WALKERS
            pbar.set_postfix(
                logP=f"{float(jnp.mean(logp)):.1f}",
                acc=f"{acc:.3f}",
                best=f"{float(jnp.max(logp)):.1f}",
            )

        if step % CHECKPOINT_EVERY == 0:
            save_checkpoint(all_positions, all_logprob, step, rng_key)

    # ── Results ──────────────────────────────────────────────────────
    chain = np.stack(all_positions, axis=0)  # (N_STEPS+1, N_WALKERS, NDIM)
    logprob = np.stack(all_logprob, axis=0)
    print(f"\nChain shape: {chain.shape}")

    if chain.shape[0] > BURNIN:
        flat_samples = chain[BURNIN:].reshape(-1, NDIM)
    else:
        flat_samples = chain.reshape(-1, NDIM)

    print(f"\n── Posterior summary ({flat_samples.shape[0]} samples) ──")
    print(f"{'Parameter':>30s} {'mean':>10s} {'std':>10s} "
          f"{'2.5%':>10s} {'97.5%':>10s} {'truth':>10s}")
    print("-" * 82)
    for i, name in enumerate(param_names):
        s = flat_samples[:, i]
        truth = res[i]
        print(f"{name:>30s} {s.mean():10.4f} {s.std():10.4f} "
              f"{np.percentile(s, 2.5):10.4f} {np.percentile(s, 97.5):10.4f} "
              f"{truth:10.4f}")

    results = {
        'chain': chain,
        'logprob': logprob,
        'flat_samples': flat_samples,
        'param_names': param_names,
    }
    with open(OUTPUT_FILE, 'wb') as f:
        pickle.dump(results, f)
    print(f"\nResults saved to {OUTPUT_FILE}")

    pd.DataFrame(flat_samples, columns=param_names).to_csv(OUTPUT_CSV, index=False)
    print(f"CSV saved to {OUTPUT_CSV}")

    return chain, logprob


if __name__ == '__main__':
    run_ensemble(resume=True)

begining logL (Array(-382.9031, dtype=float32), Array(10.781598, dtype=float32))
Initialising 16 walkers...
  Init done. logP: mean=-2163.1, max=129.5

Ensemble MCMC: 16 walkers, 1000 steps, 3D
  Stretch parameter a=2.0
  Starting from step 1


Ensemble:  13%|█▎        | 131/1000 [1:21:58<9:03:15, 37.51s/step, acc=0.062, best=585.2, logP=576.2]

In [ ]:
from nautilus import Sampler
path = '/content/drive/MyDrive/SchwarMAX-batch/'  # adjust if data lives elsewhere on Colab

ndim = 3

# Prior: mode ± half_width for each parameter
# Tune these widths: wide enough to contain the posterior,
# narrow enough to keep Nautilus efficient.
half_widths = np.array([
    0.5,    # log_L2M
    0.5,    # log_Omega
    1.0,    # log_sigma (less constrained)
])

x_mode = jnp.array([
    0, 1.5, 0.5
])
prior_low = x_mode - half_widths
prior_high = x_mode + half_widths

def prior_transform(u):
    return prior_low + (prior_high - prior_low) * u


param_names = [
    'log_LM', 'log_Omega', 'log_sigma',
]

# print(f"Prior bounds (mode ± half_width):")
# for i, name in enumerate(param_names):
#     print(f"  {name:>12}: [{prior_low[i]:.4f}, {prior_high[i]:.4f}]  (mode={x_mode[i]:.4f})")
# print()


# --- Progress tracker ---
progress_log = path + 'nautilus_progress.log'

class ProgressTracker:
    def __init__(self, log_file):
        self.n_calls = 0
        self.n_density_rejected = 0
        self.t_start = time.time()
        self.t_last_print = 0
        self.best_ll = -np.inf
        self.log_file = log_file
        header = f"{'calls':>8} | {'rejected':>8} | {'elapsed':>10} | {'best logL':>12} | {'avg s/call':>10}"
        sep = "-" * 62
        print(header)
        print(sep)
        with open(self.log_file, 'w') as f:
            f.write(header + "\n")
            f.write(sep + "\n")

    def __call__(self, params):
        ll, _ = logl_fixed_potential_bootstrap_amp(params, dict_data['dict_phi'], params_halo, dict_data, dict_data['total_bins'])
        ll_val = float(ll)
        if not np.isfinite(ll_val):
            ll_val = -1e100

        self.n_calls += 1
        if ll_val <= -1e99:
            self.n_density_rejected += 1
        if ll_val > self.best_ll:
            self.best_ll = ll_val

        t_now = time.time()
        if t_now - self.t_last_print > 60 or self.n_calls <= 5:
            elapsed = t_now - self.t_start
            avg = elapsed / self.n_calls
            rej_pct = 100 * self.n_density_rejected / self.n_calls
            line = (f"{self.n_calls:>8d} | {rej_pct:>6.1f}% | {elapsed/3600:>9.2f}h | "
                    f"{self.best_ll:>12.2f} | {avg:>9.2f}s")
            print(line, flush=True)
            with open(self.log_file, 'a') as f:
                f.write(line + "\n")
            self.t_last_print = t_now

        return ll_val

tracker = ProgressTracker(progress_log)

checkpoint_file = path + 'nautilus_checkpoint.hdf5'

sampler = Sampler(
    prior_transform,
    tracker,
    n_dim=ndim,
    n_live=500,
    n_networks=4,
    filepath=checkpoint_file,
    seed=42,
)

# Resume tracker stats from checkpoint
tracker.n_calls = sampler.n_like
tracker.t_start = time.time() - tracker.n_calls * 5  # rough estimate of prior elapsed time
print(f"Resuming from {tracker.n_calls} previous likelihood calls")

print(f"Checkpoint: {checkpoint_file}")
print(f"(Delete this file to start fresh)")
print()

t0 = time.time()
sampler.run(
    verbose=True,
    n_eff=2000,
    n_like_max=50000,
    f_live=0.05,
)
t_total = time.time() - t0

print(f"\nSampling complete in {t_total:.0f}s ({t_total/3600:.1f} hours)")
print(f"Density-rejected: {tracker.n_density_rejected}/{tracker.n_calls} "
      f"({100*tracker.n_density_rejected/max(1,tracker.n_calls):.1f}%)")

# ============================================================
# Step 4: Results
# ============================================================
print()
print("=" * 70)
print("Results")
print("=" * 70)

points, log_w, log_l = sampler.posterior()
log_Z = sampler.log_z

weights = np.exp(log_w - np.max(log_w))
weights /= weights.sum()
post_mean = np.average(points, weights=weights, axis=0)
post_std = np.sqrt(np.average((points - post_mean)**2, weights=weights, axis=0))

print(f"log Z = {log_Z:.4f}")
print(f"N_eff = {1.0 / np.sum(weights**2):.0f}")
print(f"Total logL calls: {tracker.n_calls:,}")
print()
print(f"  {'param':>12} | {'mean':>10} | {'std':>10} | {'mode':>10}")
print(f"  {'-'*42}")
for i, name in enumerate(param_names):
    print(f"  {name:>12} | {post_mean[i]:>10.4f} | {post_std[i]:>10.4f} | {x_mode[i]:>10.4f}")

# ============================================================
# Step 5: Save
# ============================================================
import pandas as pd

n_resample = 10000
idx = np.random.choice(len(points), size=n_resample, p=weights, replace=True)
equal_weight_samples = points[idx]

df = pd.DataFrame(equal_weight_samples, columns=param_names)
df['log_likelihood'] = log_l[idx]
df.to_csv(path + 'nautilus_posterior.csv', index=False)

with open(path + 'nautilus_posterior_full.pkl', 'wb') as f:
    pickle.dump({
        'points': points, 'log_w': log_w, 'log_l': log_l,
        'log_Z': log_Z, 'param_names': param_names,
        # 'prior_center': prior_center, 'prior_sigma': prior_sigma,
        # 'prior_low': prior_low, 'prior_high': prior_high,
        'x_mode': x_mode,
    }, f)
print(f"Posterior saved to: {path}nautilus_posterior.csv")



   calls | rejected |    elapsed |    best logL | avg s/call
--------------------------------------------------------------
Resuming from 0 previous likelihood calls
Checkpoint: /content/drive/MyDrive/SchwarMAX-batch/nautilus_checkpoint.hdf5
(Delete this file to start fresh)

Starting the nautilus sampler...
Please report issues at github.com/johannesulf/nautilus.
Status    | Bounds | Ellipses | Networks | Calls    | f_live | N_eff | log Z    
       1 |    0.0% |      0.01h |     -2002.37 |     45.33s
       2 |    0.0% |      0.01h |     -2002.37 |     24.43s
       3 |    0.0% |      0.01h |     -2002.37 |     17.46s
       4 |    0.0% |      0.02h |     -2002.37 |     13.99s
       5 |    0.0% |      0.02h |     -2002.37 |     11.91s
      22 |    0.0% |      0.03h |       155.11 |      5.44s
      39 |    0.0% |      0.05h |       155.11 |      4.61s
      56 |    0.0% |      0.07h |       167.41 |      4.29s
      73 |    0.0% |      0.08h |       167.41 |      4.11s
      90 |  